# ZAC 2026 LifeMate — Colab Inference (`private_test`)

Notebook chạy suy luận đầy đủ cho **625 mẫu `private_test`** trên Google Colab bằng `run_infer_batched.py` (vLLM continuous batching + lưu cache ASR & checkpoint từng mẫu lên Google Drive để `--resume` an toàn).

**Khuyến nghị GPU**:
- **L4 (24GB)** hoặc **A100 (40GB+)**: Chạy tối ưu với `max_model_len=16384`, `BATCH_SIZE=16`.
- **T4 (16GB)**: Không khuyến nghị (vì `private_test` có mẫu prompt khởi đầu ~9.7k token, T4 bị giới hạn `10240` token và dễ OOM khi chạy batch).

In [ ]:
# Bước 1: Mount Google Drive và kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

import os
import torch

!nvidia-smi

assert torch.cuda.is_available(), 'Cần bật GPU Runtime (Runtime -> Change runtime type -> L4 hoặc A100)!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

if vram_gb >= 36:
    REC_BATCH_SIZE, REC_GPU_UTIL = 24, 0.78
elif vram_gb >= 18:
    REC_BATCH_SIZE, REC_GPU_UTIL = 16, 0.72
else:
    REC_BATCH_SIZE, REC_GPU_UTIL = 4, 0.85
    print('[CẢNH BÁO] GPU < 18GB (T4): private_test có prompt dài tới ~9.7k token, nên chuyển sang L4/A100 nếu có thể!')

os.environ['ZAC_BATCH_SIZE'] = str(REC_BATCH_SIZE)
os.environ['ZAC_GPU_UTIL'] = str(REC_GPU_UTIL)
print(f'GPU: {gpu_name} ({vram_gb:.1f} GB) -> BATCH_SIZE={REC_BATCH_SIZE}, GPU_UTIL={REC_GPU_UTIL}')

In [ ]:
# Bước 2: Clone/pull repo và cài dependencies + xử lý xung đột ABI torchaudio/torchcodec
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/sun0810pi/zalo-challenge.git'
REPO_DIR = Path('/content/zalo-challenge')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'vllm>=0.6.0', 'faster-whisper>=1.1.0', 'peft>=0.14.0',
    'transformers>=4.51.0', 'accelerate>=1.0.0', 'bitsandbytes>=0.45.0',
    'soundfile>=0.12.1', 'scikit-learn>=1.3.0'
], check=True)

# Guard: vLLM thường nâng/đổi bản torch làm torchaudio/torchcodec cài sẵn trên Colab bị lỗi ABI.
# Pipeline đã đọc WAV trực tiếp bằng soundfile/wave trong src/asr.py nên không cần torchaudio/torchcodec.
broken_pkgs = []
for pkg in ('torchaudio', 'torchcodec'):
    chk = subprocess.run([sys.executable, '-c', f'import {pkg}'], capture_output=True, text=True)
    if chk.returncode != 0:
        print(f'[ABI Guard] {pkg} bị lỗi ABI với PyTorch hiện tại -> gỡ bỏ an toàn...')
        broken_pkgs.append(pkg)
if broken_pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', *broken_pkgs], check=True)

import torch, vllm, transformers, peft, faster_whisper, soundfile
print(f'OK! torch={torch.__version__}, vllm={vllm.__version__}, transformers={transformers.__version__}')

In [ ]:
# Bước 3: Cấu hình đường dẫn Drive & giải nén private_test.zip lên SSD Colab
import os
import shutil
import zipfile
from pathlib import Path

# === CHỈNH CÁC ĐƯỜNG DẪN NÀY CHO KHỚP GOOGLE DRIVE CỦA BẠN ===
DRIVE_BASE = Path('/content/drive/MyDrive/zalo-challenge')
PRIVATE_ZIP = DRIVE_BASE / 'private_test.zip'

LLM_BASE = 'Qwen/Qwen3-4B'
LLM_LORA = DRIVE_BASE / 'models' / 'final_model_v6_adapted_l4_max'  # Đặt None nếu chạy base model

# ASR mặc định: faster-whisper-large-v3-turbo-ct2 (nhanh nhất trên 3,647 clip)
# Nếu muốn dùng Whisper LoRA đã fine-tune: đặt ASR_LORA = DRIVE_BASE / 'models' / 'whisper-turbo-vi-lora'
ASR_MODEL = 'deepdml/faster-whisper-large-v3-turbo-ct2'
ASR_LORA = None

OUT_DIR = DRIVE_BASE / 'outputs_private'
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/zalo-challenge')
PRIV_DIR = REPO_DIR / 'private_test'

if not (PRIV_DIR / 'private_test.jsonl').is_file():
    assert PRIVATE_ZIP.is_file(), f'Không tìm thấy {PRIVATE_ZIP}! Hãy upload private_test.zip lên Drive.'
    print(f'Đang giải nén {PRIVATE_ZIP} -> {REPO_DIR} ...')
    tmp_extract = Path('/content/_tmp_priv_extract')
    if tmp_extract.exists():
        shutil.rmtree(tmp_extract)
    with zipfile.ZipFile(PRIVATE_ZIP, 'r') as zf:
        zf.extractall(tmp_extract)
    macosx = tmp_extract / '__MACOSX'
    if macosx.exists():
        shutil.rmtree(macosx)
    src_priv = tmp_extract / 'private_test' if (tmp_extract / 'private_test' / 'private_test.jsonl').is_file() else tmp_extract
    assert (src_priv / 'private_test.jsonl').is_file(), 'Không thấy private_test.jsonl sau khi giải nén!'
    if PRIV_DIR.exists():
        shutil.rmtree(PRIV_DIR)
    shutil.move(str(src_priv), str(PRIV_DIR))
    if tmp_extract.exists():
        shutil.rmtree(tmp_extract)

n_samples = sum(1 for line in (PRIV_DIR / 'private_test.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
assert (PRIV_DIR / 'zaic_tools').is_dir(), 'Thiếu thư mục private_test/zaic_tools!'
if LLM_LORA is not None:
    assert Path(LLM_LORA).is_dir(), f'Không tìm thấy LLM_LORA: {LLM_LORA}'
if ASR_LORA is not None:
    assert Path(ASR_LORA).is_dir(), f'Không tìm thấy ASR_LORA: {ASR_LORA}'

ASR_CACHE = OUT_DIR / ('asr_private_test_lora.json' if ASR_LORA else 'asr_private_test_ct2.json')
SUBMISSION_PATH = OUT_DIR / 'submission_private.jsonl'
SMOKE_PATH = OUT_DIR / 'submission_private_smoke5.jsonl'

os.environ['ZAC_LLM_BASE'] = str(LLM_BASE)
os.environ['ZAC_LLM_LORA'] = str(LLM_LORA) if LLM_LORA else ''
os.environ['ZAC_ASR_MODEL'] = str(ASR_MODEL)
os.environ['ZAC_ASR_LORA'] = str(ASR_LORA) if ASR_LORA else ''
os.environ['ZAC_ASR_CACHE'] = str(ASR_CACHE)
os.environ['ZAC_SUBMISSION'] = str(SUBMISSION_PATH)
os.environ['ZAC_SMOKE_OUT'] = str(SMOKE_PATH)

print(f'private_test OK: {n_samples} mẫu')
print(f'LLM LoRA: {LLM_LORA}')
print(f'ASR Cache: {ASR_CACHE}')
print(f'Output: {SUBMISSION_PATH}')

In [ ]:
# Bước 4: Smoke test nhanh 5 mẫu đầu tiên (kiểm tra ASR + LoRA + vLLM + Guardrail)
import os
import subprocess
import sys

cmd = [
    sys.executable, 'run_infer_batched.py',
    '--split', 'private_test',
    '--root', '/content/zalo-challenge',
    '--model', os.environ['ZAC_LLM_BASE'],
    '--asr-model', os.environ['ZAC_ASR_MODEL'],
    '--asr-cache-path', os.environ['ZAC_ASR_CACHE'],
    '--batch-size', os.environ['ZAC_BATCH_SIZE'],
    '--gpu-memory-utilization', os.environ['ZAC_GPU_UTIL'],
    '--limit', '5',
    '--output', os.environ['ZAC_SMOKE_OUT'],
]
if os.environ.get('ZAC_LLM_LORA'):
    cmd.extend(['--lora-path', os.environ['ZAC_LLM_LORA']])
if os.environ.get('ZAC_ASR_LORA'):
    cmd.extend(['--asr-lora-path', os.environ['ZAC_ASR_LORA']])

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd='/content/zalo-challenge', check=True)

In [ ]:
# Bước 5: Chạy full 625 mẫu private_test (có --resume tự động từ checkpoint trên Google Drive)
import os
import subprocess
import sys

cmd = [
    sys.executable, 'run_infer_batched.py',
    '--split', 'private_test',
    '--root', '/content/zalo-challenge',
    '--model', os.environ['ZAC_LLM_BASE'],
    '--asr-model', os.environ['ZAC_ASR_MODEL'],
    '--asr-cache-path', os.environ['ZAC_ASR_CACHE'],
    '--batch-size', os.environ['ZAC_BATCH_SIZE'],
    '--gpu-memory-utilization', os.environ['ZAC_GPU_UTIL'],
    '--resume',
    '--output', os.environ['ZAC_SUBMISSION'],
]
if os.environ.get('ZAC_LLM_LORA'):
    cmd.extend(['--lora-path', os.environ['ZAC_LLM_LORA']])
if os.environ.get('ZAC_ASR_LORA'):
    cmd.extend(['--asr-lora-path', os.environ['ZAC_ASR_LORA']])

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd='/content/zalo-challenge', check=True)

In [ ]:
# Bước 6: Kiểm tra chính thức bằng zaic_tools và thống kê submission_private.jsonl
import json
import os
import subprocess
import sys
from pathlib import Path

sub_path = Path(os.environ['ZAC_SUBMISSION'])
subprocess.run(
    [sys.executable, '-m', 'zaic_tools', '--split', 'private_test', '--root', '..', '--validate', str(sub_path)],
    cwd='/content/zalo-challenge/private_test',
    check=True,
)

gt_ids = [json.loads(l)['id'] for l in Path('/content/zalo-challenge/private_test/private_test.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
rows = [json.loads(l) for l in sub_path.read_text(encoding='utf-8').splitlines() if l.strip()]
sub_ids = [r['id'] for r in rows]

assert len(rows) == 625, f'Sai số dòng: {len(rows)} != 625'
assert sub_ids == gt_ids, 'Danh sách / thứ tự ID không khớp private_test.jsonl!'
assert all(isinstance(r.get('state_hash'), str) and len(r['state_hash']) == 64 for r in rows), 'Có state_hash không hợp lệ!'

doable = sum(1 for r in rows if r['tool_calls'] and not any(c['name'] == 'sys.ask_missing_info' for c in r['tool_calls']))
ask_info = sum(1 for r in rows if any(c['name'] == 'sys.ask_missing_info' for c in r['tool_calls']))
reject = sum(1 for r in rows if not r['tool_calls'])
total_calls = sum(len(r['tool_calls']) for r in rows)

print(f'VALIDATION PASS: {len(rows)}/625 mẫu, {total_calls} tool calls')
print(f'Phân bố: DOABLE={doable}, ASK_INFO={ask_info}, REJECT={reject}')
print(f'File nộp: {sub_path}')